# Shehab Big Data — Final Viva Runner V5

هذا الملف **لوحة تشغيل للمشروع الحقيقي**، وليس نسخة ثانية من الكود.

- غيّر فقط `DOCTOR_FILE` إذا أعطاك الدكتور ملفًا جديدًا.
- كل خلية تستدعي ملفات `src/` الفعلية.
- قواعد البيانات الافتراضية هنا Demo حتى لا تختلط نتائج المناقشة بقاعدة المشروع الأساسية.
- أثناء المناقشة شغّل الخلية التي يطلبها الدكتور، وليس ضروريًا `Run All`.

In [15]:
from pathlib import Path
import json, os, socket, subprocess, sys, time, urllib.request, webbrowser

PROJECT_ROOT = Path(r"D:\Projects\Shehab_BigData_Pipeline")
DOCTOR_FILE = PROJECT_ROOT / "data" / "orders_large_representative.csv"
# DOCTOR_FILE = PROJECT_ROOT / "demo" / "doctor_stress_test.csv"

VIVA_DATABASE = "shehab_viva_large_patch_test2"
# VIVA_DATABASE = "shehab_viva_large_patch_test"
# VIVA_DATABASE = "shehab_viva_large_proof"
# VIVA_DATABASE = "shehab_viva_demo"

# Large compatible CSV. The same natural Router logic is used here too.
# You may change ONLY this path to any compatible CSV larger than 200 MB.
LARGE_FILE = Path(r"D:\Projects\big_data\orders_huge_mixed_quality.csv")
SPARK_INPUT_FILE = SPARK_INPUT_FILE = PROJECT_ROOT / "data" / "orders_large_representative.csv"
# SPARK_INPUT_FILE = LARGE_FILE

# Safety for Jupyter Run All only. Set True manually when the doctor asks to
# execute a large-file Spark run. This does NOT select the engine; Router still does.
RUN_LARGE_PIPELINE = False

PYTHON = Path(r"C:\anaconda\envs\bigdata\python.exe")
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def mongo_ok():
    try:
        with socket.create_connection(("127.0.0.1", 27017), timeout=0.8):
            return True
    except OSError:
        return False

def heading(text):
    print("\n" + "=" * 76)
    print(text)
    print("=" * 76)

def run_main(*args, timeout=600):
    cmd = [str(PYTHON), "-m", "src.main", *map(str, args)]
    p = subprocess.run(
        cmd, cwd=PROJECT_ROOT, capture_output=True, text=True,
        encoding="utf-8", errors="replace", timeout=timeout
    )
    print(p.stdout)
    if p.stderr.strip():
        print(p.stderr)
    print("Exit code:", p.returncode)
    return p

heading("VIVA CONFIG")
print("Project :", PROJECT_ROOT, PROJECT_ROOT.exists())
print("Input   :", DOCTOR_FILE, DOCTOR_FILE.exists())
print("Database:", VIVA_DATABASE)
print("Python  :", PYTHON)
print("MongoDB :", "RUNNING ✅" if mongo_ok() else "STOPPED ⚠️")


VIVA CONFIG
Project : D:\Projects\Shehab_BigData_Pipeline True
Input   : D:\Projects\Shehab_BigData_Pipeline\data\orders_large_representative.csv True
Database: shehab_viva_large_patch_test2
Python  : C:\anaconda\envs\bigdata\python.exe
MongoDB : RUNNING ✅


## 1) فحص الملف فقط — بدون تخزين

**إذا قال الدكتور:** افحص الملف أولًا أو تأكد من الـSchema.

In [2]:
from src.schema import validate_csv_header

heading("SCHEMA PREFLIGHT")
check = validate_csv_header(DOCTOR_FILE)
print("Schema       : PASS ✅")
print("Columns      :", len(check.columns))
print("Reordered    :", check.reordered)
print("Canonical    :", ", ".join(check.expected_columns))


SCHEMA PREFLIGHT
Schema       : PASS ✅
Columns      : 17
Reordered    : False
Canonical    : order_id, order_date, status, customer_id, customer_name, customer_phone, customer_email, city, district, delivery_type, delivery_cost, payment_method, payment_status, payment_amount, currency, total_amount, items_json


## 2) Router الحقيقي — بدون كتابة

**إذا قال الدكتور:** اختر المحرك حسب حجم الملف.

In [3]:
from src.file_router import choose_engine, print_route
from config.settings import SMALL_FILE_THRESHOLD_MB

heading("FILE ROUTER")
ROUTE = choose_engine(DOCTOR_FILE, SMALL_FILE_THRESHOLD_MB)
print_route(ROUTE)


FILE ROUTER
File Router Decision
-------------------------------------------------------
File      : D:\Projects\Shehab_BigData_Pipeline\data\orders_large_representative.csv
Size      : 230.12 MB
Threshold : 200 MB
Engine    : pyspark
Reason    : File size 230.12 MB exceeds the 200 MB threshold.


## 3) تجهيز MongoDB

ينشئ/يتأكد من Collections + Validator + Indexes.

In [4]:
heading("MONGODB SETUP")
if not mongo_ok():
    print("MongoDB is stopped. Start-Service MongoDB as Administrator.")
else:
    from src.mongo_setup import configure_database
    configure_database(database=VIVA_DATABASE)


MONGODB SETUP
MongoDB Setup
------------------------------------------------------------
Database   : shehab_viva_large_patch_test2
Raw        : orders_raw
Validated  : orders_validated
Quarantine : quarantine_orders

orders_raw indexes:
  _id_ | unique=False
  idx_raw_run_row | unique=False
  idx_raw_ingested_at | unique=False

orders_validated indexes:
  _id_ | unique=False
  ux_validated_id_order | unique=True
  idx_validated_source_run | unique=False

quarantine indexes:
  _id_ | unique=False
  ux_quarantine_source_row | unique=True
  idx_quarantine_error_codes | unique=False

Validated schema validator: PASS
MongoDB configuration: PASS


## 4) Load Raw فقط

**إذا قال الدكتور:** خزّن البيانات أولًا قبل التنظيف.

هذه الخلية تنفذ Loader الحقيقي بحسب قرار الـRouter وتحفظ `RUN_ID`.

In [5]:
heading("RAW-FIRST LOAD")
if not mongo_ok():
    print("SKIPPED: MongoDB is stopped.")
else:
    from src.file_router import choose_engine
    from config.settings import SMALL_FILE_THRESHOLD_MB

    ROUTE = choose_engine(DOCTOR_FILE, SMALL_FILE_THRESHOLD_MB)

    if ROUTE.engine == "python_batch":
        from src.batch_loader import load_raw_with_python_batch
        RAW_METRICS = load_raw_with_python_batch(
            DOCTOR_FILE, database=VIVA_DATABASE
        )
    else:
        from src.spark_loader import load_raw_with_pyspark
        RAW_METRICS = load_raw_with_pyspark(
            DOCTOR_FILE, database=VIVA_DATABASE
        )

    RUN_ID = RAW_METRICS.get("id_run") or RAW_METRICS.get("run_id")
    print("\nRUN_ID =", RUN_ID)


RAW-FIRST LOAD
PySpark Raw Loader
-----------------------------------------------------------------
run_id      : spark-20260821T011059Z-5d96beab
File        : D:\Projects\Shehab_BigData_Pipeline\data\orders_large_representative.csv
Size MB     : 230.12
Database    : shehab_viva_large_patch_test2
Collection  : orders_raw
Spark       : 4.2.0
Input partitions: 16

Spark Raw Load Summary
-----------------------------------------------------------------
run_id          : spark-20260821T011059Z-5d96beab
loaded_raw      : 550000
input_partitions: 16
seconds_elapsed : 31.435
throughput      : 17496.58 rows/s
SPARK RAW LOAD: PASS

RUN_ID = spark-20260821T011059Z-5d96beab


## 5) إثبات Raw قبل التنظيف

يعرض عدد السجلات وسجلًا خامًا من نفس `RUN_ID`.

In [6]:
heading("RAW PROOF")
if not mongo_ok():
    print("SKIPPED")
elif "RUN_ID" not in globals():
    print("Run cell 4 first.")
else:
    from pymongo import MongoClient
    from config.settings import MONGO_URI, RAW_COLLECTION

    c = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    try:
        col = c[VIVA_DATABASE][RAW_COLLECTION]
        print("Raw count for run =", col.count_documents({"id_run": RUN_ID}))
        sample = col.find_one({"id_run": RUN_ID}, {"_id": 0})
        print(json.dumps(sample, ensure_ascii=False, indent=2, default=str))
    finally:
        c.close()


RAW PROOF
Raw count for run = 550000
{
  "id_run": "spark-20260821T011059Z-5d96beab",
  "file_source": "D:\\Projects\\Shehab_BigData_Pipeline\\data\\orders_large_representative.csv",
  "number_row_source": 0,
  "source_partition_id": 0,
  "source_record_id": 0,
  "at_ingested": "2026-08-21 01:11:22.900000",
  "engine_used": "pyspark",
  "record_raw": {
    "order_id": "طلب-100000",
    "order_date": "2025-02-24T21:29:00",
    "status": "مؤكد",
    "customer_id": "عميل-0",
    "customer_name": "محمد علي",
    "customer_phone": "702390941",
    "customer_email": "user141764@example.com",
    "city": "تعز",
    "district": "شعوب",
    "delivery_type": "سريع",
    "delivery_cost": "5000.0",
    "payment_method": "محفظة إلكترونية",
    "payment_status": "تم الدفع",
    "payment_amount": "769000.0",
    "currency": "YER",
    "total_amount": "769000.0",
    "items_json": "[{\"sku\":\"SKU-1010\",\"name\":\"هاتف سامسونج A54\",\"qty\":-2,\"unit_price\":183000.0,\"total\":549000.0},{\"sku\":\"S

## 6) Quality + Transform + Final Load

**إذا قال الدكتور:** الآن نظّف وصنّف البيانات.

يستخدم نفس `RUN_ID` الذي تم تحميله Raw في الخطوة السابقة.

In [7]:
heading("QUALITY + ELT")
if not mongo_ok():
    print("SKIPPED")
elif "RUN_ID" not in globals():
    print("Run cell 4 first.")
else:
    if ROUTE.engine == "python_batch":
        from src.elt_pipeline import run_elt
        QUALITY_METRICS = run_elt(
            run_id=RUN_ID,
            database=VIVA_DATABASE,
        )
    else:
        from src.spark_elt_pipeline import run_spark_elt
        QUALITY_METRICS = run_spark_elt(
            run_id=RUN_ID,
            database=VIVA_DATABASE,
        )

    print("\nQUALITY_METRICS:")
    print(json.dumps(QUALITY_METRICS, ensure_ascii=False, indent=2, default=str))


QUALITY + ELT
Spark ELT Quality Pipeline
----------------------------------------------------------------------
run_id             : spark-20260821T011059Z-5d96beab
database           : shehab_viva_large_patch_test2
write batch size   : 1000
shuffle partitions : 64
Mongo partitions   : 9
duplicate order IDs: 3747

Spark ELT Summary
----------------------------------------------------------------------
run_raw_count       : 550000
run_valid_count     : 381199
run_corrected_count : 99430
run_quarantine_count: 69371
classified_total     : 550000
inserted             : 480629
updated              : 0
unchanged            : 0
quarantine inserted  : 69371
quarantine existing  : 0
seconds_elapsed      : 108.338
throughput           : 5076.71 rows/s
consistency check    : PASS

Top error codes
----------------------------------------------------------------------
JSON_ITEMS_CORRUPTED: 23677
ID_CUSTOMER_MISSING: 7775
EMAIL_INVALID: 7589
ID_ORDER_DUPLICATE: 7535
VALUE_NEGATIVE_AMBIGUOUS: 7525
P

## 7) عرض Valid / Corrected / Quarantine

**دليل الدكتور المطلوب:** مثال سليم + مصحح + معزول.

In [8]:
heading("CLASSIFICATION PROOF")
if not mongo_ok():
    print("SKIPPED")
else:
    from pymongo import MongoClient
    from config.settings import MONGO_URI, VALIDATED_COLLECTION, QUARANTINE_COLLECTION

    c = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    try:
        db = c[VIVA_DATABASE]
        for status in ("valid", "corrected"):
            doc = db[VALIDATED_COLLECTION].find_one(
                {"quality_status": status}, {"_id": 0}
            )
            print(f"\n--- {status.upper()} ---")
            print(json.dumps(doc, ensure_ascii=False, indent=2, default=str))

        q = db[QUARANTINE_COLLECTION].find_one({}, {"_id": 0})
        print("\n--- QUARANTINED ---")
        print(json.dumps(q, ensure_ascii=False, indent=2, default=str))
    finally:
        c.close()


CLASSIFICATION PROOF

--- VALID ---
{
  "order_id": "طلب-100032",
  "order_date": "2025-01-09T01:36:00",
  "status": "مؤكد",
  "customer_id": "عميل-32",
  "customer_name": "يوسف طارق",
  "customer_phone": "709598591",
  "customer_email": "user601359@example.com",
  "city": "تعز",
  "district": "خور مكسر",
  "delivery_type": "سريع",
  "delivery_cost": 5000.0,
  "payment_method": "محفظة إلكترونية",
  "payment_status": "بانتظار الدفع",
  "payment_amount": 261000.0,
  "currency": "YER",
  "total_amount": 261000.0,
  "items": [
    {
      "sku": "SKU-1007",
      "name": "قرص SSD محمول",
      "qty": 2.0,
      "unit_price": 128000.0,
      "total": 256000.0
    }
  ],
  "quality_status": "valid",
  "corrections": [],
  "source_run_id": "spark-20260821T011059Z-5d96beab",
  "source_row_number": 32,
  "source_record_id": 32,
  "source_partition_id": 0,
  "source_file": "D:\\Projects\\Shehab_BigData_Pipeline\\data\\orders_large_representative.csv",
  "engine_used": "pyspark",
  "record_hash"

## 8) Collections + Unique Index + Duplicate proof

In [9]:
heading("MONGODB FINAL PROOF")
if not mongo_ok():
    print("SKIPPED")
else:
    from pymongo import MongoClient
    from config.settings import (
        MONGO_URI, RAW_COLLECTION, VALIDATED_COLLECTION, QUARANTINE_COLLECTION
    )

    c = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    try:
        db = c[VIVA_DATABASE]
        print("Raw        =", db[RAW_COLLECTION].count_documents({}))
        print("Validated  =", db[VALIDATED_COLLECTION].count_documents({}))
        print("Quarantine =", db[QUARANTINE_COLLECTION].count_documents({}))
        info = db[VALIDATED_COLLECTION].index_information()
        print("id_order unique =", info.get("ux_validated_id_order", {}).get("unique"))
        duplicates = list(db[VALIDATED_COLLECTION].aggregate([
            {"$group": {"_id": "$id_order", "n": {"$sum": 1}}},
            {"$match": {"n": {"$gt": 1}}},
        ]))
        print("Duplicate validated IDs =", len(duplicates))
    finally:
        c.close()


MONGODB FINAL PROOF
Raw        = 550000
Validated  = 480629
Quarantine = 69371
id_order unique = True
Duplicate validated IDs = 0


## 9) Metrics + Consistency

يعرض نتيجة التشغيل الحالي ومعادلة الاتساق.

In [10]:
heading("METRICS")
if "RAW_METRICS" in globals():
    print("RAW METRICS:")
    print(json.dumps(RAW_METRICS, ensure_ascii=False, indent=2, default=str))
if "QUALITY_METRICS" in globals():
    print("\nQUALITY METRICS:")
    print(json.dumps(QUALITY_METRICS, ensure_ascii=False, indent=2, default=str))
    raw = QUALITY_METRICS.get("run_raw_count")
    v = QUALITY_METRICS.get("run_valid_count")
    c = QUALITY_METRICS.get("run_corrected_count")
    q = QUALITY_METRICS.get("run_quarantine_count")
    if None not in (raw, v, c, q):
        print(f"\nConsistency: {raw} = {v} + {c} + {q} ->", "PASS ✅" if raw == v+c+q else "FAIL ❌")


METRICS
RAW METRICS:
{
  "run_id": "spark-20260821T011059Z-5d96beab",
  "engine_used": "pyspark",
  "file_name": "orders_large_representative.csv",
  "file_size_mb": 230.12,
  "loaded_raw": 550000,
  "input_partitions": 16,
  "seconds_elapsed": 31.435,
  "throughput": 17496.58
}

QUALITY METRICS:
{
  "run_id": "spark-20260821T011059Z-5d96beab",
  "database": "shehab_viva_large_patch_test2",
  "engine": "pyspark",
  "run_raw_count": 550000,
  "run_valid_count": 381199,
  "run_corrected_count": 99430,
  "run_quarantine_count": 69371,
  "classified_total": 550000,
  "inserted": 480629,
  "updated": 0,
  "unchanged": 0,
  "quarantine_inserted": 69371,
  "quarantine_existing": 0,
  "seconds_elapsed": 108.338,
  "throughput": 5076.71,
  "write_batch_size": 1000,
  "input_partitions": 9,
  "shuffle_partitions": 64,
  "duplicate_order_ids": 3747,
  "error_counts": {
    "JSON_ITEMS_CORRUPTED": 23677,
    "DATE_IMPOSSIBLE_INVALID": 3865,
    "ID_ORDER_DUPLICATE": 7535,
    "ITEMS_EMPTY": 3834,

## 10) تشغيل المشروع كاملًا بأمر واحد

هذه هي نقطة التشغيل الرئيسية التي يمكن استخدامها مع ملف الدكتور مباشرة.

In [30]:
heading("ONE MAIN COMMAND")
if mongo_ok():
    _ = run_main(
        "--input", DOCTOR_FILE,
        "--database", VIVA_DATABASE,
        timeout=600,
    )
else:
    print("SKIPPED: MongoDB is stopped.")


ONE MAIN COMMAND
Hybrid Big Data Orders Pipeline
Input    : D:\Projects\Shehab_BigData_Pipeline\data\orders_large_representative.csv
Database : shehab_viva_large_proof

CSV schema validation: PASS
Columns              : 17

File Router Decision
-------------------------------------------------------
File      : D:\Projects\Shehab_BigData_Pipeline\data\orders_large_representative.csv
Size      : 230.12 MB
Threshold : 200 MB
Engine    : pyspark
Reason    : File size 230.12 MB exceeds the 200 MB threshold.

MongoDB Setup
------------------------------------------------------------
Database   : shehab_viva_large_proof
Raw        : orders_raw
Validated  : orders_validated
Quarantine : quarantine_orders

orders_raw indexes:
  _id_ | unique=False
  idx_raw_run_row | unique=False
  idx_raw_ingested_at | unique=False

orders_validated indexes:
  _id_ | unique=False
  ux_validated_id_order | unique=True
  idx_validated_source_run | unique=False

quarantine indexes:
  _id_ | unique=False
  ux_qu

## 11) Idempotency — نفس المدخل مرتين

يستخدم قاعدة Demo مستقلة ويثبت أن `orders_validated` لا يزيد في المرة الثانية.

In [31]:
heading("IDEMPOTENCY PROOF")
IDEMP_DB = "shehab_viva_idempotency"
sample = PROJECT_ROOT / "demo" / "doctor_test_sample.csv"

if not mongo_ok():
    print("SKIPPED")
else:
    from pymongo import MongoClient
    from config.settings import MONGO_URI

    c = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    try:
        c.drop_database(IDEMP_DB)
    finally:
        c.close()

    print("FIRST RUN")
    p1 = run_main("--input", sample, "--database", IDEMP_DB, timeout=300)

    c = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    try:
        first = c[IDEMP_DB]["orders_validated"].count_documents({})
    finally:
        c.close()

    print("\nSECOND RUN — SAME INPUT")
    p2 = run_main("--input", sample, "--database", IDEMP_DB, timeout=300)

    c = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    try:
        second = c[IDEMP_DB]["orders_validated"].count_documents({})
        duplicates = list(c[IDEMP_DB]["orders_validated"].aggregate([
            {"$group": {"_id": "$id_order", "n": {"$sum": 1}}},
            {"$match": {"n": {"$gt": 1}}},
        ]))
    finally:
        c.close()

    print("Validated after first :", first)
    print("Validated after second:", second)
    print("Duplicate IDs         :", len(duplicates))
    print("IDEMPOTENCY:", "PASS ✅" if first == second and not duplicates else "FAIL ❌")


IDEMPOTENCY PROOF
FIRST RUN
Hybrid Big Data Orders Pipeline
Input    : D:\Projects\Shehab_BigData_Pipeline\demo\doctor_test_sample.csv
Database : shehab_viva_idempotency

CSV schema validation: PASS
Columns              : 17

File Router Decision
-------------------------------------------------------
File      : D:\Projects\Shehab_BigData_Pipeline\demo\doctor_test_sample.csv
Size      : 0.00 MB
Threshold : 200 MB
Engine    : python_batch
Reason    : File size 0.00 MB is less than or equal to the 200 MB threshold.

MongoDB Setup
------------------------------------------------------------
Database   : shehab_viva_idempotency
Raw        : orders_raw
Validated  : orders_validated
Quarantine : quarantine_orders

orders_raw indexes:
  _id_ | unique=False
  idx_raw_run_row | unique=False
  idx_raw_ingested_at | unique=False

orders_validated indexes:
  _id_ | unique=False
  ux_validated_id_order | unique=True
  idx_validated_source_run | unique=False

quarantine indexes:
  _id_ | unique=Fa

## 12) Upsert Update — تعديل سجل موجود بدون Duplicate

In [32]:
heading("UPSERT UPDATE PROOF")
UPDATE_DB = "shehab_viva_update"
base_file = PROJECT_ROOT / "demo" / "doctor_test_sample.csv"
update_file = PROJECT_ROOT / "demo" / "doctor_update_test.csv"

if not mongo_ok():
    print("SKIPPED")
else:
    from pymongo import MongoClient
    from config.settings import MONGO_URI

    c = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    try:
        c.drop_database(UPDATE_DB)
    finally:
        c.close()

    _ = run_main("--input", base_file, "--database", UPDATE_DB, timeout=300)
    _ = run_main("--input", update_file, "--database", UPDATE_DB, timeout=300)

    c = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    try:
        col = c[UPDATE_DB]["orders_validated"]
        count = col.count_documents({"id_order": "طلب-D1"})
        doc = col.find_one({"id_order": "طلب-D1"}, {"_id": 0, "customer_name": 1, "id_order": 1})
    finally:
        c.close()

    print("Documents with طلب-D1 =", count)
    print("Final document =", doc)
    print("UPDATE WITHOUT DUPLICATE:", "PASS ✅" if count == 1 and doc and "UPDATE TEST" in doc.get("customer_name", "") else "FAIL ❌")


UPSERT UPDATE PROOF
Hybrid Big Data Orders Pipeline
Input    : D:\Projects\Shehab_BigData_Pipeline\demo\doctor_test_sample.csv
Database : shehab_viva_update

CSV schema validation: PASS
Columns              : 17

File Router Decision
-------------------------------------------------------
File      : D:\Projects\Shehab_BigData_Pipeline\demo\doctor_test_sample.csv
Size      : 0.00 MB
Threshold : 200 MB
Engine    : python_batch
Reason    : File size 0.00 MB is less than or equal to the 200 MB threshold.

MongoDB Setup
------------------------------------------------------------
Database   : shehab_viva_update
Raw        : orders_raw
Validated  : orders_validated
Quarantine : quarantine_orders

orders_raw indexes:
  _id_ | unique=False
  idx_raw_run_row | unique=False
  idx_raw_ingested_at | unique=False

orders_validated indexes:
  _id_ | unique=False
  ux_validated_id_order | unique=True
  idx_validated_source_run | unique=False

quarantine indexes:
  _id_ | unique=False
  ux_quarantin

## 13) PySpark + Spark UI — Router طبيعي فقط

هذه الخلية **لا تجبر PySpark**. غيّر `SPARK_INPUT_FILE` فقط إلى ملف CSV متوافق حجمه أكبر من 200 MB، ثم اجعل `RUN_LARGE_PIPELINE = True`.

الـRouter الحقيقي هو الذي يجب أن يختار `pyspark`، وبعدها نعرض Jobs / Stages / Tasks / Partitions.

In [13]:
heading("SPARK UI — AUTOMATIC ROUTER")
from src.file_router import choose_engine
from config.settings import SMALL_FILE_THRESHOLD_MB

if not RUN_LARGE_PIPELINE:
    print("MANUAL SAFE MODE: لم يتم تشغيل الملف الكبير.")
    print("عند الحاجة: غيّر SPARK_INPUT_FILE فقط، ثم RUN_LARGE_PIPELINE = True.")
elif not mongo_ok():
    print("SKIPPED: MongoDB is stopped.")
elif not SPARK_INPUT_FILE.exists():
    print("Input file not found:", SPARK_INPUT_FILE)
else:
    decision = choose_engine(SPARK_INPUT_FILE, SMALL_FILE_THRESHOLD_MB)
    print("Router engine =", decision.engine)
    print("File size MB  =", decision.file_size_mb)
    print("Threshold MB  =", decision.threshold_mb)

    if decision.engine != "pyspark":
        print("STOP: هذا الملف لا يمثل ملفًا كبيرًا حسب الحد الحالي.")
        print("استخدم ملفًا أكبر من", SMALL_FILE_THRESHOLD_MB, "MB وغير المسار فقط.")
    else:
        SPARK_DB = "shehab_viva_spark"
        cmd = [
            str(PYTHON), "-m", "src.main",
            "--input", str(SPARK_INPUT_FILE),
            "--database", SPARK_DB,
        ]
        proc = subprocess.Popen(
            cmd, cwd=PROJECT_ROOT,
            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            text=True, encoding="utf-8", errors="replace",
        )

        ui_url = None
        deadline = time.time() + 60
        while time.time() < deadline and proc.poll() is None:
            for port in range(4040, 4046):
                url = f"http://127.0.0.1:{port}"
                try:
                    with urllib.request.urlopen(url, timeout=0.4) as response:
                        if response.status == 200:
                            ui_url = url
                            break
                except Exception:
                    pass
            if ui_url:
                break
            time.sleep(0.7)

        if ui_url:
            print("Spark UI =", ui_url)
            print("Show: Jobs -> Stages -> Stage details -> Tasks / Partitions")
            webbrowser.open(ui_url)
        else:
            print("Spark UI لم يظهر بعد أو انتهى التشغيل بسرعة.")

        try:
            out, err = proc.communicate(timeout=7200)
        except subprocess.TimeoutExpired:
            proc.kill()
            out, err = proc.communicate()

        print(out[-20000:])
        if err.strip():
            print(err[-5000:])
        print("Spark process exit =", proc.returncode)


SPARK UI — AUTOMATIC ROUTER
Router engine = pyspark
File size MB  = 230.12
Threshold MB  = 200
Spark UI = http://127.0.0.1:4040
Show: Jobs -> Stages -> Stage details -> Tasks / Partitions
Hybrid Big Data Orders Pipeline
Input    : D:\Projects\Shehab_BigData_Pipeline\data\orders_large_representative.csv
Database : shehab_viva_spark

CSV schema validation: PASS
Columns              : 17

File Router Decision
-------------------------------------------------------
File      : D:\Projects\Shehab_BigData_Pipeline\data\orders_large_representative.csv
Size      : 230.12 MB
Threshold : 200 MB
Engine    : pyspark
Reason    : File size 230.12 MB exceeds the 200 MB threshold.

MongoDB Setup
------------------------------------------------------------
Database   : shehab_viva_spark
Raw        : orders_raw
Validated  : orders_validated
Quarantine : quarantine_orders

orders_raw indexes:
  _id_ | unique=False
  idx_raw_run_row | unique=False
  idx_raw_ingested_at | unique=False

orders_validated in

## 14) الملف الكبير — إثبات قرار Router فقط

هذه خلية سريعة وآمنة: تفحص نفس الملف الكبير وتثبت أن Router يختار PySpark تلقائيًا، لكنها لا تعالج الملف.

In [34]:
heading("LARGE FILE ROUTER PROOF")
if LARGE_FILE.exists():
    _ = run_main("--input", LARGE_FILE, "--route-only", timeout=90)
else:
    print("Large file not found. Change LARGE_FILE in the setup cell.")


LARGE FILE ROUTER PROOF
Hybrid Big Data Orders Pipeline
Input    : D:\Projects\big_data\orders_huge_mixed_quality.csv
Database : shehab_bigdata

CSV schema validation: PASS
Columns              : 17

File Router Decision
-------------------------------------------------------
File      : D:\Projects\big_data\orders_huge_mixed_quality.csv
Size      : 12650.32 MB
Threshold : 200 MB
Engine    : pyspark
Reason    : File size 12650.32 MB exceeds the 200 MB threshold.
ROUTE ONLY: PASS

Exit code: 0


## 15) Advanced Path B — اختياري للفردي / مطلوب لمجموعة طالبين

Initial + Delta Update/Insert + إعادة نفس Delta.

In [35]:
heading("ADVANCED PATH B")
INC_DB = "shehab_incremental_demo"
initial = PROJECT_ROOT / "demo" / "incremental_initial.csv"
delta = PROJECT_ROOT / "demo" / "incremental_delta_v2.csv"

print("Initial command:")
print(f'{PYTHON} -m src.main --input "{initial}" --database "{INC_DB}"')
print("\nDelta command:")
print(f'{PYTHON} -m src.incremental_loader --input "{delta}" --database "{INC_DB}"')
print("\nRe-run the same Delta to prove Idempotency.")


ADVANCED PATH B
Initial command:
C:\anaconda\envs\bigdata\python.exe -m src.main --input "D:\Projects\Shehab_BigData_Pipeline\demo\incremental_initial.csv" --database "shehab_incremental_demo"

Delta command:
C:\anaconda\envs\bigdata\python.exe -m src.incremental_loader --input "D:\Projects\Shehab_BigData_Pipeline\demo\incremental_delta_v2.csv" --database "shehab_incremental_demo"

Re-run the same Delta to prove Idempotency.


## 16) results.json

In [36]:
heading("RESULTS.JSON")
path = PROJECT_ROOT / "reports" / "results.json"
if path.exists():
    data = json.loads(path.read_text(encoding="utf-8-sig"))
    print(json.dumps(data, ensure_ascii=False, indent=2, default=str)[-30000:])
else:
    print("results.json not found.")


RESULTS.JSON
{
  "project": "Shehab Big Data Hybrid Orders Pipeline",
  "small_batch_run": {
    "engine": "python_batch",
    "rows": 100000,
    "valid": 69310,
    "corrected": 18122,
    "quarantine": 12568,
    "consistency": "PASS"
  },
  "spark_100k_validation": {
    "engine": "pyspark",
    "rows": 100000,
    "valid": 69310,
    "corrected": 18122,
    "quarantine": 12568,
    "consistency": "PASS"
  },
  "large_raw_load": {
    "engine": "pyspark",
    "file_size_gb": 12.35,
    "rows": 30000000,
    "input_partitions": 99,
    "seconds": 221.956,
    "throughput_rows_per_sec": 135161.79,
    "status": "PASS"
  },
  "large_elt": {
    "rows": 30000000,
    "valid": 20785463,
    "corrected": 5421631,
    "quarantine": 3792906,
    "classified_total": 30000000,
    "duplicate_order_ids": 207689,
    "mongo_partitions": 444,
    "shuffle_partitions": 256,
    "seconds": 3985.264,
    "throughput_rows_per_sec": 7527.73,
    "consistency": "PASS"
  },
  "idempotency_test": {
  

## 17) الاختبارات

In [16]:
heading("PYTEST")

from uuid import uuid4

# نجعل Pytest يستخدم مجلد Temp داخل المشروع بدل Temp الخاص بويندوز
pytest_temp = PROJECT_ROOT / ".viva_tmp" / f"pytest_{uuid4().hex[:8]}"
pytest_temp.parent.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["TEMP"] = str(PROJECT_ROOT / ".viva_tmp")
env["TMP"] = str(PROJECT_ROOT / ".viva_tmp")

p = subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pytest",
        str(PROJECT_ROOT / "tests"),
        "-q",
        f"--basetemp={pytest_temp}",
    ],
    cwd=PROJECT_ROOT,
    env=env,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    timeout=240,
)

print(p.stdout)

if p.stderr.strip():
    print(p.stderr)

print("Exit code:", p.returncode)

if p.returncode == 0:
    print("PYTEST: PASS ✅")
else:
    print("PYTEST: FAIL ❌")


PYTEST
.........................................                                [100%]
41 passed in 0.34s

Exit code: 0
PYTEST: PASS ✅


## قاعدة المناقشة

مع **أي ملف** من الدكتور نغيّر المسار فقط ونستخدم نفس الأمر الطبيعي:

```powershell
python -m src.main --input "D:\path\doctor_orders.csv" --database "shehab_doctor_demo"
```

- إذا كان الحجم `<= 200 MB` يختار Router: `python_batch`.
- إذا كان الحجم `> 200 MB` يختار Router: `pyspark`.
- لا يوجد في النسخة النهائية خيار لإجبار المحرك يدويًا.
